In [218]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [219]:
sber = pd.read_csv('data/SBRF.txt')

In [220]:
def good_dataframe(data, timeframe):
  """Преобразует сырые рыночные данные в чистый DataFrame с правильными типами и индексом времени
    
    Подготавливает данные для технического анализа.
    
    Args:
        data (pd.DataFrame): Исходный DataFrame с рыночными данными, содержащий столбцы:
            ['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<VOL>']
            
    Returns:
        tuple: Возвращает кортеж из двух DataFrame:
            - Основной DataFrame
            - Копия DataFrame для безопасного резервирования
            
    Processing Logic:
        1. Удаление избыточных столбцов
        2. Переименование столбцов в human-friendly формат
        3. Преобразование типов данных
        4. Создание правильного временного индекса
    """
  # 1. Удаляем ненужные столбцы (тикер и период не нужны для анализа)
  data.drop(['<TICKER>', '<PER>'], inplace=True, axis=1)
    
  # 2. Переименовываем столбцы для удобства работы
  data.columns = ['date', 'time', 'open', 'high', 'low', 'close', 'volume']
    
  # 3. Преобразуем дату из формата YYYYMMDD в datetime
  data['date'] = pd.to_datetime(data['date'], format='%Y%m%d')
    
  # 4. Обрабатываем время (HHMMSS -> datetime.time)
  data['time'] = pd.to_datetime(data['time'], format='%H%M%S').dt.time
    
  # 5. Комбинируем дату и время в единую метку времени
  data['time'] = pd.to_datetime(
        data['date'].astype('str') + ' ' + data['time'].astype('str'))
    
  # 6. Удаляем отдельный столбец даты (теперь он в индексе)
  data.drop(['date'], inplace=True, axis=1)
  
  # 7. Установка индекса
  data_final = data.set_index('time')
  
  
  
  def new_timeframe(data, timeframe):
    """Преобразует минутные данные (1М) в указанный временной интервал, сохраняя структуру OHLCV-данных.
    
    Использует принципы агрегации свечных данных:
    - Open - первое значение периода
    - High - максимум периода
    - Low - минимум периода
    - Close - последнее значение периода
    - Volume - сумма объема за период

    Args:
        data (pd.DataFrame): Исходный DataFrame с 1-минутными данными, 
                            должен содержать колонки ['open', 'high', 'low', 'close', 'volume']
                            и иметь DateTimeIndex
        timeframe (str): Желаемый таймфрейм из списка доступных:
                        ['5 min', '15 min', '30 min', '1h', '2h', '4h', 'D']

    Returns:
        pd.DataFrame: Новый DataFrame с преобразованными данными в указанном таймфрейме
        
    Raises:
        ValueError: Если передан неподдерживаемый timeframe
    """
    dict_tf = {'5 min' : '5min', '15 min' : '15min', '30 min' : '30min',
               '1h' : '1h', '2h' : '2h', '4h' : '4h', 'D' : 'D'}

    return_data = data.resample(dict_tf[timeframe]).agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }).dropna()
    
    return return_data
  
  result = new_timeframe(data_final, timeframe)
  
  return result


In [221]:
sber_15_bad = good_dataframe(sber, '15 min')
sber_15 = sber_15_bad.reset_index()
sber_15

,time,open,high,low,close,volume
0,2009-01-11 10:30:00,2301.0,2346.0,2265.0,2340.0,2993
1,2009-01-11 10:45:00,2341.0,2380.0,2340.0,2361.0,4804
2,2009-01-11 11:00:00,2366.0,2369.0,2352.0,2361.0,1660
3,2009-01-11 11:15:00,2360.0,2362.0,2324.0,2347.0,2811
4,2009-01-11 11:30:00,2343.0,2350.0,2338.0,2346.0,425
...,...,...,...,...,...,...
229931,2025-06-30 22:45:00,29853.0,29856.0,29834.0,29838.0,212
229932,2025-06-30 23:00:00,29837.0,29840.0,29825.0,29828.0,266
229933,2025-06-30 23:15:00,29827.0,29865.0,29827.0,29849.0,276
229934,2025-06-30 23:30:00,29849.0,29853.0,29845.0,29845.0,166


Напишем функцию детекции свечного паттерна

In [222]:
def detection_bullish_engulfing_pattern(data, filtr='NO'):
    """
    Функция для детекции бычьего паттерна 'поглощение' (Bullish Engulfing) на ценовых данных.
    Функция позволяет применять дополнительный фильтр для подтверждения надежности паттерна.

    Параметры:
    ----------
    data : pd.DataFrame
        DataFrame с ценовыми данными, должен содержать колонки:
        - 'open': цены открытия
        - 'close': цены закрытия
        - 'high', 'low', 'volume' для расширенных фильтров
        
    filtr : str, optional, default='NO'
        Тип фильтра для подтверждения паттерна:
        
        Базовые фильтры:
        - 'NO' - детекция паттерна без дополнительных фильтров
        
        Фильтры подтверждения бычьим движением:
        - '2_grow_candle_after_pattern' - паттерн подтверждается 2 растущими свечами после него

    Возвращает:
    -----------
    pd.DataFrame
        Исходный DataFrame с добавленной колонкой:
        - 'pattern': бинарный indicator (0/1), где 1 обозначает наличие паттерна
        - 'signal': бинарный indicator (0/1), где 1 обозначает сигнал на вход(следующая свеча после паттерна), создано для облегчения проверки стратегий 
    """
    data = data.copy()
    data['pattern'] = 0
    data['signal'] = 0
    data['strategy'] = 'bullish_engulfing_pattern'
    
    # Векторизованные вычисления
    body = data['close'] - data['open']
    
    # Базовое условие бычьего поглощения
    base_condition = (
        (body < 0) & 
        (body.shift(-1) > 0) &
        (data['open'] < data['close'].shift(-1)) & 
        (data['close'] > data['open'].shift(-1)))
    
    if filtr == 'NO':
        # Отмечаем 2 свечи паттерна
        pattern_mask = base_condition
        data.loc[pattern_mask, 'pattern'] = 1
        data.loc[pattern_mask.shift(1).fillna(False), 'pattern'] = 1
        # Сигнал - следующая свеча после завершения паттерна
        data.loc[pattern_mask.shift(2).fillna(False), 'signal'] = 1
        

    elif filtr == '2_grow_candle_after_pattern':
        # 4 свечи: паттерн + 2 подтверждения
        pattern_mask = base_condition & (body.shift(-2) > 0) & (body.shift(-3) > 0)
        data.loc[pattern_mask, 'pattern'] = 1
        data.loc[pattern_mask.shift(1).fillna(False), 'pattern'] = 1
        data.loc[pattern_mask.shift(2).fillna(False), 'pattern'] = 1
        data.loc[pattern_mask.shift(3).fillna(False), 'pattern'] = 1
        # Сигнал - после второй подтверждающей свечи
        data.loc[pattern_mask.shift(4).fillna(False), 'signal'] = 1
        

    data['filtr'] = filtr
    
    return data
        


In [223]:
def detection_bullish_engulfing_pattern(data, filtr='NO'):
    data = data.copy()
    data['pattern'] = 0
    data['signal'] = 0
    data['strategy'] = 'bullish_engulfing_pattern'
    
    body = data['close'] - data['open']
    
    # Базовое условие
    base_condition = (
        (body < 0) & 
        (body.shift(-1) > 0) &
        (data['open'] < data['close'].shift(-1)) & 
        (data['close'] > data['open'].shift(-1))
    )
    
    # 🔑 КЛЮЧЕВОЕ УПРОЩЕНИЕ: запрет перекрытий
    pattern_start = base_condition & (~base_condition.shift(1).fillna(False))
    
    if filtr == 'NO':
        # Помечаем 2 свечи: t и t+1
        data.loc[pattern_start, 'pattern'] = 1
        data.loc[pattern_start.shift(1).fillna(False), 'pattern'] = 1
        # Сигнал — в t+2 (следующая свеча после паттерна)
        data.loc[pattern_start.shift(2).fillna(False), 'signal'] = 1

    elif filtr == '2_grow_candle_after_pattern':
        # Проверяем подтверждение ТОЛЬКО для непересекающихся паттернов
        confirmed = pattern_start & (body.shift(-2) > 0) & (body.shift(-3) > 0)
        
        data.loc[confirmed, 'pattern'] = 1
        data.loc[confirmed.shift(1).fillna(False), 'pattern'] = 1
        data.loc[confirmed.shift(2).fillna(False), 'pattern'] = 1
        data.loc[confirmed.shift(3).fillna(False), 'pattern'] = 1
        data.loc[confirmed.shift(4).fillna(False), 'signal'] = 1

    data['filtr'] = filtr
    return data

In [224]:
df = detection_bullish_engulfing_pattern(sber_15, filtr='NO')

In [225]:
df

,time,open,high,low,close,volume,pattern,signal,strategy,filtr
0,2009-01-11 10:30:00,2301.0,2346.0,2265.0,2340.0,2993,0,0,bullish_engulfing_pattern,NO
1,2009-01-11 10:45:00,2341.0,2380.0,2340.0,2361.0,4804,0,0,bullish_engulfing_pattern,NO
2,2009-01-11 11:00:00,2366.0,2369.0,2352.0,2361.0,1660,0,0,bullish_engulfing_pattern,NO
3,2009-01-11 11:15:00,2360.0,2362.0,2324.0,2347.0,2811,0,0,bullish_engulfing_pattern,NO
4,2009-01-11 11:30:00,2343.0,2350.0,2338.0,2346.0,425,0,0,bullish_engulfing_pattern,NO
...,...,...,...,...,...,...,...,...,...,...
229931,2025-06-30 22:45:00,29853.0,29856.0,29834.0,29838.0,212,0,0,bullish_engulfing_pattern,NO
229932,2025-06-30 23:00:00,29837.0,29840.0,29825.0,29828.0,266,1,0,bullish_engulfing_pattern,NO
229933,2025-06-30 23:15:00,29827.0,29865.0,29827.0,29849.0,276,1,0,bullish_engulfing_pattern,NO
229934,2025-06-30 23:30:00,29849.0,29853.0,29845.0,29845.0,166,0,1,bullish_engulfing_pattern,NO


Напишем функцию, которая считает прибыль или убыток при входе на цене открытия после завершения паттерна, держит позицию N следующих свечей, где N пробегает [1, 2, ..., 10, 15, 20, ..., 50] и выходит на закрытии последней свечи.

In [226]:
def lst_trade_statistics_bul(data, commission=0.001):
   # Удалим сперва все отметки паттерна и сигнала с последних 51 строки
   data.loc[len(data) - 51 : len(data), ['pattern', 'signal']] = 0
   # Проходимся циклом по периодам удержания позиции
   all_strategy = []
   for N in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 35, 40, 45, 50]:
      strategy = [] # Список для результатов стратегии
      # Блок для определения цены входа и выхода
      exit_candidate = data['close'].shift(-N)
      is_entry = data['signal'] == 1
      entry_price = data['close'][is_entry]
      exit_price = exit_candidate[is_entry]
      # Считаем результат
      profit = exit_price - entry_price
      if len(profit) == 0:
         strategy_name = data['strategy'][0]
         filtr = data['filtr'][0]
         all_strategy.append({'strategy' : strategy_name, 'filtr' : filtr, 'N': N, 'trades': 0, 'win_rate': 0.0, 'total_profit': 0.0, 'results': 0, 'profit_factor' : 0})
         continue
      # Чистая прибыль
      profit_net = np.where(profit >= 0, profit * (1 - commission), profit * (1 + commission))
      # Статистика
      profit_factor = (np.sum(profit_net > 0) / (np.sum(profit_net < 0)))
      # Названия
      strategy_name = data['strategy'][0]
      filtr = data['filtr'][0]
      
      all_strategy.append({'strategy' : strategy_name, 'filtr' : filtr,
                           'N' : N,
                           'trades' : len(profit_net),
                           'win_rate' : np.round(np.mean(profit_net > 0), 2),
                           'total_profit' : np.round(np.sum(profit_net), 1),
                           'results' : profit_net,
                           'profit_factor' : np.round(profit_factor, 2)})
   return all_strategy

result = lst_trade_statistics_bul(df)
result
      

[{'strategy': 'bullish_engulfing_pattern',
  'filtr': 'NO',
  'N': 1,
  'trades': 7634,
  'win_rate': 0.48,
  'total_profit': -5857.0,
  'results': array([ -3.003,  -5.005,  -6.006, ...,  -9.009, -20.02 ,   6.993]),
  'profit_factor': 0.96},
 {'strategy': 'bullish_engulfing_pattern',
  'filtr': 'NO',
  'N': 2,
  'trades': 7634,
  'win_rate': 0.49,
  'total_profit': -3906.2,
  'results': array([-11.011,  -3.003,   5.994, ...,  -3.003, -27.027,  40.959]),
  'profit_factor': 1.0},
 {'strategy': 'bullish_engulfing_pattern',
  'filtr': 'NO',
  'N': 3,
  'trades': 7634,
  'win_rate': 0.49,
  'total_profit': -2343.4,
  'results': array([-14.014,  10.989,  15.984, ...,  -2.002,  25.974,  12.987]),
  'profit_factor': 0.98},
 {'strategy': 'bullish_engulfing_pattern',
  'filtr': 'NO',
  'N': 4,
  'trades': 7634,
  'win_rate': 0.5,
  'total_profit': 4036.6,
  'results': array([-16.016,  10.989,  23.976, ...,   0.999,  51.948,  24.975]),
  'profit_factor': 1.01},
 {'strategy': 'bullish_engulfing_pa

In [227]:
def shift_features_2_candle(data):
    data_c = data.copy()
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern', 'time']:
        data_c[f"{i}_N"] = data[i]
        data_c[f'{i}_N-1'] = data[i].shift(1)
    data_c.drop(['open', 'close', 'low', 'high', 'volume', 'pattern', 'time'], axis=1, inplace=True)
    data_c.dropna(inplace=True)
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern']:
        data_c[f'{i}_N-1'] = data_c[f'{i}_N-1'].astype('int32')
    return data_c


In [228]:
# Делаем двойной датасет
df_cl = shift_features_2_candle(df)
# Делаем 1 столбец паттерн

In [230]:
df_cl['pattern'] = np.where((df_cl['pattern_N'] == 1) & (df_cl['pattern_N-1'] == 1), 1, 0)

In [231]:
len(df_cl[df_cl['pattern_N'] == 1])


15268

In [232]:
df_cl

,signal,strategy,filtr,open_N,open_N-1,close_N,close_N-1,low_N,low_N-1,high_N,high_N-1,volume_N,volume_N-1,pattern_N,pattern_N-1,time_N,time_N-1,pattern
1,0,bullish_engulfing_pattern,NO,2341.0,2301,2361.0,2340,2340.0,2265,2380.0,2346,4804,2993,0,0,2009-01-11 10:45:00,2009-01-11 10:30:00,0
2,0,bullish_engulfing_pattern,NO,2366.0,2341,2361.0,2361,2352.0,2340,2369.0,2380,1660,4804,0,0,2009-01-11 11:00:00,2009-01-11 10:45:00,0
3,0,bullish_engulfing_pattern,NO,2360.0,2366,2347.0,2361,2324.0,2352,2362.0,2369,2811,1660,0,0,2009-01-11 11:15:00,2009-01-11 11:00:00,0
4,0,bullish_engulfing_pattern,NO,2343.0,2360,2346.0,2347,2338.0,2324,2350.0,2362,425,2811,0,0,2009-01-11 11:30:00,2009-01-11 11:15:00,0
5,0,bullish_engulfing_pattern,NO,2346.0,2343,2346.0,2346,2344.0,2338,2355.0,2350,773,425,0,0,2009-01-11 11:45:00,2009-01-11 11:30:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229931,0,bullish_engulfing_pattern,NO,29853.0,29864,29838.0,29845,29834.0,29845,29856.0,29868,212,224,0,0,2025-06-30 22:45:00,2025-06-30 22:30:00,0
229932,0,bullish_engulfing_pattern,NO,29837.0,29853,29828.0,29838,29825.0,29834,29840.0,29856,266,212,0,0,2025-06-30 23:00:00,2025-06-30 22:45:00,0
229933,0,bullish_engulfing_pattern,NO,29827.0,29837,29849.0,29828,29827.0,29825,29865.0,29840,276,266,0,0,2025-06-30 23:15:00,2025-06-30 23:00:00,0
229934,0,bullish_engulfing_pattern,NO,29849.0,29827,29845.0,29849,29845.0,29827,29853.0,29865,166,276,0,0,2025-06-30 23:30:00,2025-06-30 23:15:00,0


In [233]:
mask = (df_cl['pattern_N-1']==1) & (df_cl['pattern_N']==1)
len(df_cl[mask])

7897

In [234]:
len(df_cl[df_cl['pattern_N-1'] == 1])

15268

In [ ]:
df_cl.drop(['signal', 'pattern_N', 'pattern_N-1'], axis=1, inplace=True)

In [ ]:
df_cl

,strategy,filtr,open_N,open_N-1,close_N,close_N-1,low_N,low_N-1,high_N,high_N-1,volume_N,volume_N-1,time_N,time_N-1,pattern
1,bullish_engulfing_pattern,NO,2341.0,2301,2361.0,2340,2340.0,2265,2380.0,2346,4804,2993,2009-01-11 10:45:00,2009-01-11 10:30:00,0
2,bullish_engulfing_pattern,NO,2366.0,2341,2361.0,2361,2352.0,2340,2369.0,2380,1660,4804,2009-01-11 11:00:00,2009-01-11 10:45:00,0
3,bullish_engulfing_pattern,NO,2360.0,2366,2347.0,2361,2324.0,2352,2362.0,2369,2811,1660,2009-01-11 11:15:00,2009-01-11 11:00:00,0
4,bullish_engulfing_pattern,NO,2343.0,2360,2346.0,2347,2338.0,2324,2350.0,2362,425,2811,2009-01-11 11:30:00,2009-01-11 11:15:00,0
5,bullish_engulfing_pattern,NO,2346.0,2343,2346.0,2346,2344.0,2338,2355.0,2350,773,425,2009-01-11 11:45:00,2009-01-11 11:30:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229931,bullish_engulfing_pattern,NO,29853.0,29864,29838.0,29845,29834.0,29845,29856.0,29868,212,224,2025-06-30 22:45:00,2025-06-30 22:30:00,0
229932,bullish_engulfing_pattern,NO,29837.0,29853,29828.0,29838,29825.0,29834,29840.0,29856,266,212,2025-06-30 23:00:00,2025-06-30 22:45:00,0
229933,bullish_engulfing_pattern,NO,29827.0,29837,29849.0,29828,29827.0,29825,29865.0,29840,276,266,2025-06-30 23:15:00,2025-06-30 23:00:00,0
229934,bullish_engulfing_pattern,NO,29849.0,29827,29845.0,29849,29845.0,29827,29853.0,29865,166,276,2025-06-30 23:30:00,2025-06-30 23:15:00,0


In [235]:
len(datas.loc[10, 'results'])

7634

In [236]:
len(df_cl[df_cl['pattern']==1])

7897

In [237]:
df_cl['profit'] = 0
df_cl.loc[df_cl['pattern']==1, 'profit'] = datas.loc[10, 'results']

ValueError: Must have equal len keys and value when setting with an iterable

Реализация функции для проверки качества скрипта написана, однако её можно подулучшить, но это чуть позже

Теперь перейдем к классификации